# Extractor de conocimiento de Prig

Lee libros enteros con un modelo grande y produce un `.prigpack` por libro, que
luego se importa en Prig para razonar en local.

**Antes de ejecutar nada**, en el panel de la derecha:

| Ajuste | Valor |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (solo para instalar vLLM; después ya no hace falta) |
| Persistence | *Files only*, si vas a continuar en otra sesión |

Y añade tres datasets con **+ Add Input**:

| Dataset | Qué es | Ruta que queda |
|---|---|---|
| `prig-job` | la carpeta que genera Prig (`chunks.jsonl`, `manifest.json`, `outline.json`) | `/kaggle/input/prig-job` |
| `prig-worker` | `prig_extract.py`, `extraction_schema.py`, `domains.py` | `/kaggle/input/prig-worker` |
| el modelo | los pesos descargados de HuggingFace | `/kaggle/input/<tu-modelo>` |

Subir el modelo como dataset no es un capricho: evita volver a bajar 9 GB en cada
sesión y permite trabajar con internet desactivado.

## 1 · Comprobar la máquina que te ha tocado

Kaggle no siempre asigna lo mismo. Esto tarda segundos y evita descubrir a mitad de la corrida que solo hay una GPU.

In [ ]:
import subprocess, os, json, glob

print(subprocess.run(["nvidia-smi",
      "--query-gpu=index,name,memory.total,compute_cap",
      "--format=csv"], capture_output=True, text=True).stdout)

import torch
print("GPUs visibles para torch:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
print()
print("La T4 es sm_75 (Turing): sin bfloat16 y sin FlashAttention-2.")
print("Por eso el extractor fuerza fp16; no lo cambies.")

## 2 · Comprobar las entradas

Un dataset mal enlazado da un error a los veinte minutos, cuando el modelo ya está cargado. Mejor saberlo ahora.

In [ ]:
JOB    = "/kaggle/input/prig-job"          # ajusta a tus nombres de dataset
WORKER = "/kaggle/input/prig-worker"
MODELO = "/kaggle/input/qwen25-14b-awq"
SALIDA = "/kaggle/working/packs"

for etiqueta, ruta in [("trabajo", JOB), ("worker", WORKER), ("modelo", MODELO)]:
    existe = os.path.isdir(ruta)
    print(f"{'OK ' if existe else 'FALTA'} {etiqueta:<9} {ruta}")
    if existe:
        print("      ", sorted(os.listdir(ruta))[:6])

# Algunos datasets quedan dentro de una subcarpeta; esto lo detecta
if os.path.isdir(JOB) and not os.path.isfile(f"{JOB}/chunks.jsonl"):
    hallados = glob.glob(f"{JOB}/**/chunks.jsonl", recursive=True)
    if hallados:
        JOB = os.path.dirname(hallados[0])
        print(f"\nchunks.jsonl estaba un nivel más abajo; uso: {JOB}")

manifiesto = json.load(open(f"{JOB}/manifest.json"))
print(f"\n{manifiesto['total_books']} libros · {manifiesto['total_chunks']} fragmentos")
for b in manifiesto["books"]:
    print(f"  [{b['domain']:<16}] {b['title'][:46]:<48} {b['chunks']:>5} fragmentos")

## 3 · Instalar vLLM

Es lo único que necesita internet. Si la instalación rompe el `torch` que Kaggle
trae de serie, reinicia el kernel y vuelve a ejecutar desde la celda 1: los
datasets siguen montados.

In [ ]:
!pip install -q vllm 2>&1 | tail -5
import vllm; print("vLLM", vllm.__version__)

## 4 · El sondeo, antes de gastar cuota

Cincuenta fragmentos bastan para saber si el modelo sirve. Lo que mide es la
**tasa de verificación de citas**: qué porcentaje de lo que afirma el modelo
aparece de verdad, literalmente, en el fragmento.

| Tasa | Qué hacer |
|---|---|
| > 85 % | adelante |
| 60-85 % | aceptable, lo demás se descarta solo |
| < 60 % | cambia de modelo; seguir es tirar horas de GPU |

Fíjate también en los minutos estimados: multiplicados por el total de fragmentos
te dicen si la corrida entra en una sesión o hay que partirla.

In [ ]:
!python {WORKER}/prig_extract.py \
    --input  {JOB} \
    --model  {MODELO} \
    --output /kaggle/working/sondeo \
    --engine vllm --quantization awq --tensor-parallel 2 \
    --batch 16 --limit 50

## 5 · La corrida completa

Dos formas de usar las dos T4. **No son equivalentes.**

### A · Reparto entre las dos GPU (recomendado)

Dos procesos independientes, uno por GPU, cada uno con la mitad de los fragmentos.
Las T4 de Kaggle están unidas por PCIe, no por NVLink, así que repartir el trabajo
rinde más que repartir el modelo — siempre que el modelo quepa en una sola GPU
(un 14B en AWQ ocupa ~9 GB de los 15 disponibles).

### B · Un solo modelo partido entre las dos (`--tensor-parallel 2`)

Necesario solo si el modelo no cabe en una GPU (32B). Paga el coste de comunicación
en cada capa.

In [ ]:
# --- A · Reparto entre las dos GPU ---
import subprocess, time

procesos = []
for gpu in (0, 1):
    procesos.append(subprocess.Popen(
        ["python", f"{WORKER}/prig_extract.py",
         "--input", JOB, "--model", MODELO, "--output", SALIDA,
         "--engine", "vllm", "--quantization", "awq",
         "--gpu", str(gpu),            # cada proceso ve UNA sola GPU
         "--shard", f"{gpu}/2",        # y procesa la mitad de los fragmentos
         "--batch", "32", "--max-len", "4096"],
        stdout=open(f"/kaggle/working/gpu{gpu}.log", "w"),
        stderr=subprocess.STDOUT))
    time.sleep(20)   # cargar los dos modelos a la vez satura la lectura del disco

print("Dos procesos en marcha. Sigue el avance con la celda siguiente.")

In [ ]:
# Avance de ambas GPU (vuelve a ejecutar esta celda cuando quieras)
for gpu in (0, 1):
    print(f"=== GPU {gpu} " + "=" * 50)
    !tail -4 /kaggle/working/gpu{gpu}.log
    print()

In [ ]:
# Esperar a que terminen las dos
for i, p in enumerate(procesos):
    p.wait()
    print(f"GPU {i} terminó con código {p.returncode}")

## 6 · Empaquetar

**Este paso es obligatorio tras un reparto.** Cada proceso tiene solo su mitad de
los fragmentos; si empaquetara por su cuenta produciría un `.prigpack` que parece
correcto y trae medio libro. Por eso el extractor no empaqueta al acabar un reparto
y hay que unir los parciales aquí.

In [ ]:
!python {WORKER}/prig_extract.py \
    --input {JOB} --model {MODELO} --output {SALIDA} --pack-only

## 7 · Revisar antes de descargar

Si algún libro sale incompleto, no lo importes: relanza el reparto que falte y vuelve a empaquetar.

In [ ]:
import zipfile, json, os

filas = []
for nombre in sorted(os.listdir(SALIDA)):
    if not nombre.endswith(".prigpack"):
        continue
    m = json.loads(zipfile.ZipFile(f"{SALIDA}/{nombre}").read("manifest.json"))
    verificadas = m["claims_verified"]
    tasa = verificadas / max(1, verificadas + m.get("claims_rejected", 0)) * 100
    filas.append((m.get("complete", True), m["title"], m["domain"],
                  m["chunks_processed"], m.get("chunks_expected", "?"),
                  verificadas, tasa, os.path.getsize(f"{SALIDA}/{nombre}") / 1e6))

print(f"{'':3} {'libro':<40} {'campo':<17} {'frag.':>10} {'afirm.':>7} {'citas':>6} {'MB':>6}")
for ok, titulo, dom, proc, esp, cl, tasa, mb in filas:
    print(f"{'OK ' if ok else '!! '} {titulo[:40]:<40} {dom:<17} "
          f"{proc:>4}/{str(esp):<5} {cl:>7} {tasa:>5.0f}% {mb:>6.1f}")

incompletos = [f for f in filas if not f[0]]
print()
print(f"{len(filas) - len(incompletos)}/{len(filas)} libros completos")
if incompletos:
    print("INCOMPLETOS — no los importes todavía:")
    for f in incompletos:
        print("   ", f[1])

## 8 · Descargar

Los `.prigpack` están en `/kaggle/working/packs`. Descárgalos desde el panel
**Output** (o haz *Save Version* para conservarlos).

Ya en tu equipo:

1. Déjalos en `~/.prig_books/packs/`
2. Abre Prig → **Biblioteca** → bloque *Base de conocimiento* → **Importar**

Eso verifica los checksums, vuelca cada libro, une los conceptos entre libros,
precomputa los dosieres y te enseña el informe libro a libro.

---

## Si la sesión se corta

El extractor escribe `_parcial*.jsonl` sobre la marcha y **reanuda** donde se quedó.
Para que sobreviva a la sesión:

1. *Save Version* → conserva `/kaggle/working`
2. En la sesión siguiente, añade esa salida como dataset de entrada
3. Copia los parciales antes de relanzar:

```python
!mkdir -p {SALIDA} && cp /kaggle/input/<version-anterior>/packs/_parcial*.jsonl {SALIDA}/
```

Con muchos libros conviene partir el trabajo en varios cuadernos de 20-30 libros:
depender de una sola sesión larga es la forma más fácil de perder horas de cuota.

## Ajustes cuando algo va mal

| Síntoma | Ajuste |
|---|---|
| CUDA out of memory | `--batch 16` (o 8), o `--gpu-util 0.85` |
| Muy lento | sube `--batch`; baja `--max-len` a 4096 |
| Salidas cortadas a media frase | sube `--max-new-tokens` |
| Tasa de citas baja | `--profile core`, o un modelo mejor |
| Solo una GPU disponible | quita el reparto y usa un único proceso sin `--gpu` |